<div class="alert alert-block">
<b>Step 2:</b> Feature Engineering</div>

_Imports_

In [1]:
import pandas as pd
import geopandas as gpd # (.shp)
import matplotlib.pyplot as plt
import rasterio # (.tif)
from rasterio.mask import mask
from tqdm import tqdm
import os
import glob
import seaborn as sns
import numpy as np
import pyodbc
from rasterstats import zonal_stats
import pickle
from shapely.geometry import Point 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from collections import Counter 
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, accuracy_score, f1_score, classification_report
from sklearn.metrics import confusion_matrix


_Loading_

In [3]:
save_path = "data/processed/df_encoded.pkl"

with open(save_path, "rb") as f:
       df_encoded = pickle.load(f)   

This is feature engineering / feature selection, not basic preprocessing.

removing highly correlated variables in soil

After data preprocessing, a feature selection step was conducted to reduce redundancy and multicollinearity.

Based on the correlation analysis, variables with high correlation (|r| ≥ 0.7) were identified and removed to reduce redundancy and multicollinearity. Specifically:

TEXTURE_USDA_*, REF_BULK, and SILT were removed as they were highly correlated with SAND, CLAY, and COARSE (r = 0.79 to 0.92).

TOTAL_N and CEC_SOIL were removed due to very high correlation with ORG_CARBON (r = 0.87–0.92).

TEB was removed because it strongly correlated with CEC_EFF (r = 0.95).

ESP was removed because it correlated with ELEC_COND (r = 0.79).

BULK was removed because of high correlation with PH_WATER (r = 0.84).
This ensures the dataset contains independent, physically meaningful features for fire prediction while minimizing redundancy.

In [4]:
df_encoded.columns

Index(['lat', 'lon', 'fire', 'prec_winter', 'prec_spring', 'prec_summer',
       'prec_fall', 'tmax_winter', 'tmax_spring', 'tmax_summer', 'tmax_fall',
       'tmin_winter', 'tmin_spring', 'tmin_summer', 'tmin_fall', 'elevation',
       'COARSE', 'SAND', 'SILT', 'CLAY', 'BULK', 'REF_BULK', 'ORG_CARBON',
       'PH_WATER', 'TOTAL_N', 'CN_RATIO', 'CEC_SOIL', 'CEC_CLAY', 'CEC_EFF',
       'TEB', 'BSAT', 'ALUM_SAT', 'ESP', 'TCARBON_EQ', 'GYPSUM',
       'TEXTURE_SOTER_-', 'TEXTURE_SOTER_C', 'TEXTURE_SOTER_F',
       'TEXTURE_SOTER_M', 'TEXTURE_SOTER_nan', 'ELEC_COND_0.0',
       'ELEC_COND_1.0', 'ELEC_COND_14.0', 'ELEC_COND_16.0', 'ELEC_COND_2.0',
       'ELEC_COND_32.0', 'ELEC_COND_nan', 'TEXTURE_USDA_10.0',
       'TEXTURE_USDA_11.0', 'TEXTURE_USDA_12.0', 'TEXTURE_USDA_3.0',
       'TEXTURE_USDA_5.0', 'TEXTURE_USDA_7.0', 'TEXTURE_USDA_9.0',
       'TEXTURE_USDA_Unknown', 'TEXTURE_USDA_nan', 'LCCCode_0003 / 0004',
       'LCCCode_0004 // 0003', 'LCCCode_0010', 'LCCCode_0011',
       'LCCC

In [5]:
cols_to_remove = [
    # Redundant texture / soil
    'TEXTURE_USDA_10.0', 'TEXTURE_USDA_11.0', 'TEXTURE_USDA_12.0',
    'TEXTURE_USDA_3.0', 'TEXTURE_USDA_5.0', 'TEXTURE_USDA_7.0',
    'TEXTURE_USDA_9.0', 'TEXTURE_USDA_Unknown', 'TEXTURE_USDA_nan',
    'REF_BULK', 'SILT',
    
    # Organic matter / N
    'TOTAL_N', 'CEC_SOIL',
    
    # Bases / cations
    'TEB',
    
    # Salinity
    'ESP',
    
    # Bulk density
    'BULK'
]

df_cleaned = df_encoded.drop(columns=cols_to_remove)


## Calculating vif

The Variance Inflation Factor (VIF) is a statistical measure used to detect multicollinearity among predictor variables in a dataset. Multicollinearity occurs when a feature can be predicted from other features, which can distort model estimates and reduce interpretability, especially in linear or logistic regression models.

How it works:

For each numeric feature 
𝑋
𝑖
X
i
	​

, VIF measures how well it can be linearly predicted from all other numeric features.

VIF is calculated as:

VIF
(
𝑋
𝑖
)
=
1
1
−
𝑅
𝑖
2
VIF(X
i
	​

)=
1−R
i
2
	​

1
	​


where 
𝑅
𝑖
2
R
i
2
	​

 is the coefficient of determination when 
𝑋
𝑖
X
i
	​

 is regressed on all other predictors.

Interpretation:

1	No correlation with other features
1–5	Moderate correlation; usually acceptable
>5 or 10	High correlation; consider removing or combining features

Why we used VIF:

Even after removing highly correlated variables based on the correlation matrix, some numeric features may still be redundant with multiple other features.

VIF provides a quantitative and model-aware way to identify these features.

By removing or combining features with high VIF, we reduce multicollinearity, simplify the model, and maintain stable and interpretable predictions for fire occurrence.

In this study:

We calculated VIF for all numeric soil and environmental variables.

Features with high VIF were considered for removal if they were physically redundant with other variables.

This ensures the final dataset contains informative, independent predictors for fire prediction.

In [10]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

# ---------------------------
# Step 1: Select numeric soil features
# ---------------------------
soil_numeric_features = [
    'COARSE', 'SAND', 'CLAY', 
    'ORG_CARBON', 'PH_WATER', 'CEC_EFF', 
    'TCARBON_EQ', 'GYPSUM', 'ALUM_SAT', 'BSAT'
]

X_soil = df_cleaned[soil_numeric_features]

# ---------------------------
# Step 2: Add a constant column for statsmodels regression
# ---------------------------
X_soil_with_const = sm.add_constant(X_soil)  # adds intercept term

# ---------------------------
# Step 3: Initialize a dataframe to store VIF values
# ---------------------------
vif_data = pd.DataFrame(columns=["Feature", "VIF", "R_squared"])

# ---------------------------
# Step 4: Calculate VIF for each feature
# ---------------------------
for i in range(1, X_soil_with_const.shape[1]):  # start from 1 to skip the constant
    feature_name = X_soil_with_const.columns[i]
    
    # Variance Inflation Factor
    vif_value = variance_inflation_factor(X_soil_with_const.values, i)
    
    # Compute R^2 of this feature predicted by all others
    y = X_soil_with_const.iloc[:, i]  # target feature
    X_others = X_soil_with_const.drop(X_soil_with_const.columns[i], axis=1)  # all other features
    model = sm.OLS(y, X_others).fit()
    r_squared = model.rsquared
    
    # Store results
    vif_data = pd.concat([vif_data, pd.DataFrame({
        "Feature": [feature_name],
        "VIF": [vif_value],
        "R_squared": [r_squared]
    })], ignore_index=True)

# ---------------------------
# Step 5: Display VIF values
# ---------------------------
print(vif_data.sort_values(by="VIF", ascending=False))



C:\Users\dell\AppData\Local\Temp\ipykernel_17900\1874297976.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  vif_data = pd.concat([vif_data, pd.DataFrame({


      Feature        VIF  R_squared
5     CEC_EFF  16.609502   0.939793
1        SAND  12.253946   0.918394
7      GYPSUM  11.971655   0.916469
4    PH_WATER  10.920640   0.908430
2        CLAY   8.415631   0.881173
6  TCARBON_EQ   5.506148   0.818385
3  ORG_CARBON   3.510472   0.715138
9        BSAT   2.720376   0.632404
0      COARSE   2.495178   0.599227
8    ALUM_SAT   1.451588   0.311100


In [12]:
df_cleaned[soil_numeric_features].skew()

COARSE         0.851281
SAND          -0.111376
CLAY           2.683611
ORG_CARBON     4.167587
PH_WATER      -1.633744
CEC_EFF        3.079996
TCARBON_EQ     0.123925
GYPSUM         4.858372
ALUM_SAT      17.537545
BSAT          -1.717973
dtype: float64

In [15]:

pd.set_option('display.max_columns', None)  # show all columns
pd.set_option('display.max_rows', None)     # show all rows
pd.set_option('display.width', 200)         # set display width
pd.set_option('display.float_format', '{:.4f}'.format)  # format floats

In [16]:
# Skewness of numeric columns
numeric_features = df_cleaned.select_dtypes(include=['float64', 'float32', 'int64']).columns
skewness = df_cleaned[numeric_features].skew().sort_values(ascending=False)
print(skewness)


LCCCode_21496-121340 // 21497-129401    654.9496
LCCCode_21497-15045                     247.5425
LCCCode_6020                            142.9117
LCCCode_11490 // 11494                   89.9479
LCCCode_21454 // 21446 // 21450          83.8401
LCCCode_21499-121340                     52.7491
LCCCode_Unknown                          30.3560
ELEC_COND_16.0                           28.7248
LCCCode_21518                            24.1623
ELEC_COND_32.0                           23.9816
LCCCode_21497-121340                     20.6285
LCCCode_7001 // 8001                     17.6884
ALUM_SAT                                 17.5375
LCCCode_21450                            12.4641
LCCCode_21446 // 21450-121340 / 21454    12.0896
LCCCode_11498                             8.8979
LCCCode_0010                              8.1065
TEXTURE_SOTER_F                           6.9266
LCCCode_0003 / 0004                       5.8535
LCCCode_20058                             5.3383
ELEC_COND_0.0       

## Feature Engineering

1️⃣ Interaction features

In [18]:
df_cleaned['CLAY_SAND_RATIO'] = df_cleaned['CLAY'] / (df_cleaned['SAND'] + 1e-5)


some fire risks depend on soil type, not just individual percentages. Ratios summarize this information.

3️⃣ Encoding categorical features

already done

Log / Box-Cox transformations

In [20]:
df_cleaned['GYPSUM_log'] = np.log1p(df_cleaned['GYPSUM'])

In [21]:
output_dir = "data/processed/"
os.makedirs(output_dir, exist_ok=True)

# Sauvegarder df_cleaned
df_cleaned_path = os.path.join(output_dir, "df_cleaned.pkl")
df_cleaned.to_pickle(df_cleaned_path)